In [1]:
import time
from tqdm import tqdm
import pandas as pd
import dask.dataframe as dd
import hashlib
import re
import os


In [2]:
# Смотрим первые 20 строк первого файла
print("="*50)
print("USERS_RT.csv - первые 20 строк:")
print("="*50)
sample_rt = pd.read_csv('downloads/USERS_RT.csv', nrows=20)
print(sample_rt)
print("\n" + "="*50)
print("Колонки:", sample_rt.columns.tolist())
print("="*50)



USERS_RT.csv - первые 20 строк:
          id               last_login  is_superuser              first_name  \
0   35333796  2025-11-03 09:05:15.487         False  двойник Григория Лепса   
1   36805864                      NaN         False                     NaN   
2   69596994  2025-12-25 11:20:08.161         False                     NaN   
3   37212100  2024-08-19 19:06:00.103         False                     NaN   
4   41474085                      NaN         False                     NaN   
5   22897346  2024-09-02 01:22:53.407         False                     NaN   
6   35164085  2025-10-18 15:39:12.194         False                     NaN   
7   20723027  2025-12-07 17:55:56.010         False                 Ishmael   
8   68723109                      NaN         False                     NaN   
9   52822817  2025-12-29 14:24:29.279         False                     NaN   
10  46729859                      NaN         False                     NaN   
11  43130510  2024-1

In [3]:
# Смотрим первые 20 строк второго файла
print("\n"*2)
print("="*50)
print("USERS_PREM.csv - первые 20 строк:")
print("="*50)
sample_prem = pd.read_csv('downloads/USERS_PREM.csv', nrows=20)
print(sample_prem)
print("\n" + "="*50)
print("Колонки:", sample_prem.columns.tolist())
print("="*50)




USERS_PREM.csv - первые 20 строк:
         id        username                  gid_id  locked  enabled  \
0   9316095     79048779909  2sTUwgC9uvXDwoVnUh7m19   False     True   
1   9316096     79139776473  8mznkcE5zeZJvM7fdTnMYm   False     True   
2   9316097     79151788816  PBdmBKStixZhAP6u8SjBxi   False     True   
3   9316098     79053237005  ALncU5NkRC2AEoYoXQnLf9   False     True   
4   9316099  62895000716100  2ZVS8yJ8GfXxGttP1euCwH   False     True   
5   9316100     79068415848  F3qz7qqhmpMJfhMii5hr6B   False     True   
6   9316101     79100992905  GGSD2BcbnLtXJkByoxgNpZ   False     True   
7   9316102     79692144638  5D7mEnBVZvq15jMwGfCSzB   False     True   
8   9316103    821057926073  K4CfF7SfYDexQkKo5AbKcu   False     True   
9   9316104     79119419800  DG1J4oj99ivADDokBuSarp   False     True   
10  9316105  62895000606681  J9HgSmcfZZ8YN5FXxMPsKa   False     True   
11  9316106     79140706075  HzCLyoEcNuLxHqYcKPkPte   False     True   
12  9316107     79026453051

In [4]:
def normalize_phone(phone):
    """Приводит телефон к единому формату (10 цифр)"""
    if pd.isna(phone):
        return None
    
    phone_str = str(phone).strip()
    digits = re.sub(r'\D', '', phone_str)
    
    if not digits:
        return None
    
    # Нормализация
    if len(digits) == 11 and digits[0] in ['7', '8']:
        digits = digits[1:]
    elif len(digits) == 12 and digits[:2] in ['79', '78']:
        digits = digits[2:]
    
    return digits if len(digits) == 10 else None

def mask_phone(phone):
    """Детерминированное хеширование"""
    normalized = normalize_phone(phone)
    if normalized is None:
        return None
    return hashlib.sha256(normalized.encode()).hexdigest()[:12]

In [5]:
# ============================================
# ОБРАБОТКА РУТУБА (USERS_RT)
# ============================================
print("="*50)
print("Обработка USERS_RT (RuTube)")
print("="*50)

chunk_size = 100000
first_chunk = True

# Колонки для рутуба: user_id, squirrel_id, masked_phone
useful_cols_rt = ['user_id', 'squirrel_id']  # masked_phone добавим отдельно

with tqdm(desc="Маскировка RT", unit=" строк") as pbar:
    for chunk in pd.read_csv('downloads/USERS_RT.csv', chunksize=chunk_size, low_memory=False):
        
        # Маскируем телефон
        chunk['masked_phone'] = chunk['phone_number'].apply(mask_phone)
        
        # Берем нужные колонки
        available_cols = [col for col in useful_cols_rt if col in chunk.columns]
        result_chunk = chunk[available_cols + ['masked_phone']]
        
        # Сохраняем
        result_chunk.to_csv('users_rt_masked.csv', 
                          mode='a' if not first_chunk else 'w',
                          header=first_chunk,
                          index=False)
        
        first_chunk = False
        pbar.update(len(chunk))

print("✓ RT обработан\n")

Обработка USERS_RT (RuTube)


Маскировка RT: 69026666 строк [04:06, 279592.83 строк/s]

✓ RT обработан



In [6]:
# ============================================
# ОБРАБОТКА ПРЕМЬЕРА (USERS_PREM) - ИСПРАВЛЕННЫЙ
# ============================================
print("="*50)
print("Обработка USERS_PREM (Премьер)")
print("="*50)

def get_phone_from_premier(row):
    """Берем телефон из subscriber_id или username"""
    for col in ['subscriber_id', 'username']:
        if col in row and pd.notna(row[col]):
            phone = row[col]
            if normalize_phone(phone):
                return phone
    return None

first_chunk = True
chunk_size = 100000

# Колонки для премьера: id, gid_id, masked_phone
useful_cols_prem = ['id', 'gid_id']

print("Пробуем читать с пропуском проблемных строк...")

with tqdm(desc="Маскировка PREM", unit=" строк") as pbar:
    for chunk in pd.read_csv('downloads/USERS_PREM.csv', 
                             chunksize=chunk_size, 
                             on_bad_lines='skip',      # Пропускаем проблемные строки
                             engine='python'):         # Python engine (без low_memory)
        
        # Получаем телефон
        chunk['raw_phone'] = chunk.apply(get_phone_from_premier, axis=1)
        chunk['masked_phone'] = chunk['raw_phone'].apply(mask_phone)
        chunk = chunk.drop('raw_phone', axis=1)
        
        # Берем нужные колонки
        available_cols = [col for col in useful_cols_prem if col in chunk.columns]
        result_chunk = chunk[available_cols + ['masked_phone']]
        
        # Сохраняем
        result_chunk.to_csv('users_prem_masked.csv',
                          mode='a' if not first_chunk else 'w',
                          header=first_chunk,
                          index=False)
        
        first_chunk = False
        pbar.update(len(chunk))

print("✓ PREM обработан\n")

Обработка USERS_PREM (Премьер)
Пробуем читать с пропуском проблемных строк...


Маскировка PREM: 30496701 строк [06:02, 84110.61 строк/s]

✓ PREM обработан



In [7]:
# ============================================
# СТАТИСТИКА
# ============================================
print("="*50)
print("СТАТИСТИКА СОЗДАННЫХ ФАЙЛОВ")
print("="*50)

import os

for file in ['users_rt_masked.csv', 'users_prem_masked.csv']:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024**3
        df = pd.read_csv(file)
        print(f"\n{file}:")
        print(f"  - Размер: {size:.2f} ГБ")
        print(f"  - Строк: {len(df):,}")
        print(f"  - Колонки: {df.columns.tolist()}")
        if 'masked_phone' in df.columns:
            print(f"  - Уникальных masked_phone: {df['masked_phone'].nunique():,}")
            print(f"  - Пустых телефонов: {df['masked_phone'].isna().sum():,}")

СТАТИСТИКА СОЗДАННЫХ ФАЙЛОВ

users_rt_masked.csv:
  - Размер: 1.17 ГБ
  - Строк: 69,026,666
  - Колонки: ['squirrel_id', 'masked_phone']
  - Уникальных masked_phone: 41,903,929
  - Пустых телефонов: 27,115,921

users_prem_masked.csv:
  - Размер: 1.25 ГБ
  - Строк: 30,496,701
  - Колонки: ['id', 'gid_id', 'masked_phone']
  - Уникальных masked_phone: 29,023,021
  - Пустых телефонов: 1,458,027


In [8]:
billing_file_1 = 'downloads/billing_rutube_24_25.csv'  
billing_file_2 = 'downloads/billing_premier_24_25.csv'  

print("="*50)
print("ДИАГНОСТИКА БИЛИНГОВЫХ ТАБЛИЦ")
print("="*50)

# Первый билинговый файл
print("\n📊 БИЛЛИНГ 1")
print("-"*30)
try:
    # Проверяем существует ли файл
    if not os.path.exists(billing_file_1):
        print(f"❌ Файл не найден: {billing_file_1}")
        print(f"   Текущая директория: {os.getcwd()}")
        print(f"   Файлы в директории: {os.listdir('.')[:10]}...")
    else:
        size = os.path.getsize(billing_file_1) / 1024**3
        print(f"✓ Размер: {size:.2f} ГБ")
        
        # Читаем первые 3 строки
        sample = pd.read_csv(billing_file_1, nrows=3)
        print(f"✓ Колонки ({len(sample.columns)}): {sample.columns.tolist()}")
        print(f"\nПример данных (первые 3 строки):")
        print(sample)
        
        # Проверяем наличие ID для матчинга
        rt_keys = ['squirrel_id', 'user_id']
        prem_keys = ['id', 'gid_id']
        
        found_rt = [k for k in rt_keys if k in sample.columns]
        found_prem = [k for k in prem_keys if k in sample.columns]
        
        if found_rt:
            print(f"\n🔑 Найдены ключи для Рутуба: {found_rt}")
        if found_prem:
            print(f"\n🔑 Найдены ключи для Премьера: {found_prem}")
        if not found_rt and not found_prem:
            print(f"\n⚠️ Не найдены стандартные ключи. Какие колонки можно использовать для матчинга?")
            
except Exception as e:
    print(f"❌ Ошибка: {e}")

# Второй билинговый файл
print("\n" + "="*50)
print("📊 БИЛЛИНГ 2")
print("-"*30)
try:
    if not os.path.exists(billing_file_2):
        print(f"❌ Файл не найден: {billing_file_2}")
    else:
        size = os.path.getsize(billing_file_2) / 1024**3
        print(f"✓ Размер: {size:.2f} ГБ")
        
        sample = pd.read_csv(billing_file_2, nrows=3)
        print(f"✓ Колонки ({len(sample.columns)}): {sample.columns.tolist()}")
        print(f"\nПример данных (первые 3 строки):")
        print(sample)
        
        # Проверяем наличие ID для матчинга
        rt_keys = ['squirrel_id', 'user_id']
        prem_keys = ['id', 'gid_id']
        
        found_rt = [k for k in rt_keys if k in sample.columns]
        found_prem = [k for k in prem_keys if k in sample.columns]
        
        if found_rt:
            print(f"\n🔑 Найдены ключи для Рутуба: {found_rt}")
        if found_prem:
            print(f"\n🔑 Найдены ключи для Премьера: {found_prem}")
            
except Exception as e:
    print(f"❌ Ошибка: {e}")

print("\n" + "="*50)
print("ГОТОВО")
print("="*50)

ДИАГНОСТИКА БИЛИНГОВЫХ ТАБЛИЦ

📊 БИЛЛИНГ 1
------------------------------
✓ Размер: 1.76 ГБ
✓ Колонки (80): ['passport_id', 'subscriber_id', 'phone', 'subscription_id', 'user_id', 'reg_date', 'start_ts', 'end_ts', 'start_date', 'end_date', 'kind', 'money', 'sub_type', 'product_id', 'product_code', 'tariff_id_num', 'tariff_id', 'shop', 'vendor_code', 'payment_system', 'promo_campaign', 'is_multi', 'is_recovered', 'is_grace_period', 'auto_renewal', 'frozen', 'is_main', 'payment_type', 'card_type', 'card_expiry_month', 'card_expiry_year', 'card_last4', 'issuer_name', 'rrn', 'transaction_id', 'created_at', 'new', 'renewal', 'returned', 'inserted', 'inflow', 'outflow', 'sub_status', 'returned30', 'inflow30', 'outflow30', 'sub_status30', 'is_last', 'is_active', 'lt_total', 'lt_continouos', 'ltv', 'ltv_cycle', 'outflow_period', 'cnt_outflow', 'cnt_total_subs', 'cnt_sub', 'cnt_trial', 'cnt_sub_discount', 'cnt_promo', 'cnt_b2b_retail', 'cnt_external', 'shifted_end_date', 'next_start_date', 'nex

In [9]:
print("="*60)
print("МАСКИРОВКА ТЕЛЕФОНОВ В БИЛИНГОВЫХ ТАБЛИЦАХ")
print("="*60)

# Загружаем справочники
print("\nЗагрузка справочников...")
rt_dict = pd.read_csv('users_rt_masked.csv')  # squirrel_id, masked_phone
prem_dict = pd.read_csv('users_prem_masked.csv')  # id, gid_id, masked_phone

print(f"Рутуб справочник: {len(rt_dict):,} строк")
print(f"Премьер справочник: {len(prem_dict):,} строк")

# ============================================
# 1. ОБРАБОТКА БИЛИНГА РУТУБА
# ============================================
print("\n" + "="*50)
print("БИЛИНГ РУТУБА")
print("="*50)

chunk_size = 100000
first_chunk = True

for chunk in tqdm(pd.read_csv('downloads/billing_rutube_24_25.csv', chunksize=chunk_size),
                  desc="Маскировка"):
    
    # Добавляем masked_phone через join по user_id
    chunk = chunk.merge(rt_dict[['squirrel_id', 'masked_phone']], 
                        left_on='user_id', 
                        right_on='squirrel_id', 
                        how='left')
    
    # Удаляем служебную колонку и оригинальные телефоны
    chunk = chunk.drop(['squirrel_id', 'subscriber_id', 'phone'], axis=1)
    
    # Сохраняем
    chunk.to_csv('billing_rutube_masked.csv', 
                 mode='a' if not first_chunk else 'w',
                 header=first_chunk,
                 index=False)
    
    first_chunk = False

print("✓ Готово")

# ============================================
# 2. ОБРАБОТКА БИЛИНГА ПРЕМЬЕРА
# ============================================
print("\n" + "="*50)
print("БИЛИНГ ПРЕМЬЕРА")
print("="*50)

first_chunk = True

for chunk in tqdm(pd.read_csv('downloads/billing_premier_24_25.csv', chunksize=chunk_size),
                  desc="Маскировка"):
    
    # Добавляем masked_phone через join по user_id
    chunk = chunk.merge(prem_dict[['id', 'masked_phone']], 
                        left_on='user_id', 
                        right_on='id', 
                        how='left')
    
    # Удаляем служебную колонку и оригинальные телефоны
    chunk = chunk.drop(['id', 'subscriber_id', 'phone'], axis=1)
    
    # Сохраняем
    chunk.to_csv('billing_premier_masked.csv', 
                 mode='a' if not first_chunk else 'w',
                 header=first_chunk,
                 index=False)
    
    first_chunk = False

print("✓ Готово")

# ============================================
# 3. ПРОВЕРКА РЕЗУЛЬТАТОВ
# ============================================
print("\n" + "="*50)
print("ПРОВЕРКА РЕЗУЛЬТАТОВ")
print("="*50)

import os

for file in ['billing_rutube_masked.csv', 'billing_premier_masked.csv']:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024**3
        df_sample = pd.read_csv(file, nrows=2)
        print(f"\n📁 {file}")
        print(f"   Размер: {size:.2f} ГБ")
        print(f"   Колонки: {df_sample.columns.tolist()}")
        
        # Проверяем, что телефонов больше нет
        if 'subscriber_id' in df_sample.columns or 'phone' in df_sample.columns:
            print(f"   ⚠️ ВНИМАНИЕ: Остались колонки с телефонами!")
        else:
            print(f"   ✅ Колонки с телефонами удалены")
        
        if 'masked_phone' in df_sample.columns:
            print(f"   ✅ masked_phone добавлена")

МАСКИРОВКА ТЕЛЕФОНОВ В БИЛИНГОВЫХ ТАБЛИЦАХ

Загрузка справочников...
Рутуб справочник: 69,026,666 строк
Премьер справочник: 30,496,701 строк

БИЛИНГ РУТУБА


/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (20,33) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (20,33) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: Dt

✓ Готово

БИЛИНГ ПРЕМЬЕРА


/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (13,20,27,28,32,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (20,27,28,32,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (13,20) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (13,20) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.13/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (13,20) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/opt/anaconda3/lib/python3.1

✓ Готово

ПРОВЕРКА РЕЗУЛЬТАТОВ

📁 billing_rutube_masked.csv
   Размер: 1.65 ГБ
   Колонки: ['passport_id', 'subscription_id', 'user_id', 'reg_date', 'start_ts', 'end_ts', 'start_date', 'end_date', 'kind', 'money', 'sub_type', 'product_id', 'product_code', 'tariff_id_num', 'tariff_id', 'shop', 'vendor_code', 'payment_system', 'promo_campaign', 'is_multi', 'is_recovered', 'is_grace_period', 'auto_renewal', 'frozen', 'is_main', 'payment_type', 'card_type', 'card_expiry_month', 'card_expiry_year', 'card_last4', 'issuer_name', 'rrn', 'transaction_id', 'created_at', 'new', 'renewal', 'returned', 'inserted', 'inflow', 'outflow', 'sub_status', 'returned30', 'inflow30', 'outflow30', 'sub_status30', 'is_last', 'is_active', 'lt_total', 'lt_continouos', 'ltv', 'ltv_cycle', 'outflow_period', 'cnt_outflow', 'cnt_total_subs', 'cnt_sub', 'cnt_trial', 'cnt_sub_discount', 'cnt_promo', 'cnt_b2b_retail', 'cnt_external', 'shifted_end_date', 'next_start_date', 'next_kind', 'diff_day', 'first_start_date', 'f

In [10]:
print('пук')

пук


In [11]:
files_to_export = [
    'billing_rutube_masked.csv',
    'billing_premier_masked.csv'
]

for file in files_to_export:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024**3
        print(f"\n✅ {file}")
        print(f"   Размер: {size:.2f} ГБ")
        print(f"   Путь: {os.path.abspath(file)}")
    else:
        print(f"\n❌ {file} не найден")

print("\n" + "="*60)
print("ФАЙЛЫ ГОТОВЫ К ИСПОЛЬЗОВАНИЮ")
print("="*60)


✅ billing_rutube_masked.csv
   Размер: 1.65 ГБ
   Путь: /Users/mbohonko/billing_rutube_masked.csv

✅ billing_premier_masked.csv
   Размер: 1.92 ГБ
   Путь: /Users/mbohonko/billing_premier_masked.csv

ФАЙЛЫ ГОТОВЫ К ИСПОЛЬЗОВАНИЮ
